In [1]:
# ============================================================
# Tablas_por_enlace_h (celda de Jupyter) — v1.1  (NIVEL HORARIO, T+1 h)
# Versión horaria de Tablas_por_enlace v1.9:
#   - Para cada ventana W (24, 168): Arch1–4 (A y B), ARIMA_t+1 y Chronos con contexto C = W
#     (salida de Test_Chronos_h.py).
#   - Instantes fijos por (W, enlace): Realidad > TRIGGER y predicción de las 8 arquitecturas
#     (MODOS_MASCARA), ARIMA y Chronos. No dependen de MODOS.
#   - TABLAS POR ENLACE: Link | CV% | PeakRatio | P99 | ARIMA | Chronos | Arch1–4 (una por
#     métrica, W y modo). Mejor de cada enlace en negrita. GLOBAL: media de los enlaces
#     (MAE, RMSE, MAPE); WAPE = Σ|e| / Σ real.
#   - RESUMEN: filas W × métrica, columnas ARIMA, Chronos y Arch1–4 (A y B), como la tabla
#     resumen de minutos con ω = 0.
#   - Formato: > 9999 → 1.3e4 ; 999 … 9999 → entero ; < 999 → un decimal (redondeo .5 hacia arriba).
#   - Notebook (HTML, negrita) y, con GUARDAR_TEX = True, .tex en DIR_SALIDA.
# ============================================================
import os
import numpy as np
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP
from IPython.display import display, HTML


DATASET = "brain"
ROOT_RESULTS, TRIGGER = "Results_horas", 1000

    
DIR_BASE      = f"{ROOT_RESULTS}/results_baselines_1h"
CARPETA       = "results_Arq{ID}_1h_{MODO}_W{W}"
DIR_CHRONOS   = "results_chronos-bolt-small_C{W}_1h"      # dentro de ROOT_RESULTS
COL_ARIMA     = "ARIMA_t+1"
COL_PRED      = "Prediccion"
IDS           = [1, 2, 3, 4]
VENTANAS      = [24, 168]
MODOS         = ["A", "B"]
MODOS_MASCARA = ["A", "B"]
METRICAS      = ["MAE", "RMSE", "MAPE", "WAPE"]
QUITAR_INTRA  = False          # quita los pares X_X (intra-nodo)
DIR_SALIDA    = f"Tablas_latex_h_{DATASET}" + ("_sin_intra" if QUITAR_INTRA else "")
GUARDAR_TEX   = True
HACER_POR_ENLACE = True
HACER_RESUMEN    = True


def sci(v):
    if pd.isna(v):
        return "--"
    m, e = f"{v:.2e}".split("e")
    return f"{m}e{int(e)}"


def _red(x, dec):
    return Decimal(repr(float(x))).quantize(Decimal(1).scaleb(-dec), rounding=ROUND_HALF_UP)


def val(v):
    if pd.isna(v):
        return "--"
    if v > 9999:
        e = int(np.floor(np.log10(v)))
        m = _red(v / 10 ** e, 1)
        if m >= 10:
            m, e = _red(m / 10, 1), e + 1
        return f"{m}e{e}"
    if v >= 999:
        return str(int(_red(v, 0)))
    return f"{_red(v, 1)}"


def ruta_arq(ID, W, MODO, e):
    n = CARPETA.format(ID=ID, W=W, MODO=MODO)
    return os.path.join(ROOT_RESULTS, n, f"test_{n.removeprefix('results_')}_{e}.csv")


def ruta_chronos(W, e):
    d = DIR_CHRONOS.format(W=W)
    return os.path.join(ROOT_RESULTS, d, f"test_{d.removeprefix('results_')}_{e}.csv")


def enlaces_de(ID, W, MODO):
    d = os.path.join(ROOT_RESULTS, CARPETA.format(ID=ID, W=W, MODO=MODO))
    if not os.path.isdir(d):
        print(f"⚠️ falta {d}")
        return set()
    pref = "test_" + os.path.basename(d).removeprefix("results_") + "_"
    return {f[len(pref):-4] for f in os.listdir(d) if f.startswith(pref) and f.endswith(".csv")}


def leer_pred(f, nombre):
    return pd.read_csv(f, parse_dates=["timestamp"]).set_index("timestamp")[[COL_PRED]].rename(
        columns={COL_PRED: nombre})


_CACHE = {}


def instantes(W, e):
    """Instantes fijos para (W, enlace): real > TRIGGER y predicción de todas las arquitecturas de
    MODOS_MASCARA, ARIMA y Chronos (C = W)."""
    if (W, e) in _CACHE:
        return _CACHE[(W, e)]
    idx = None
    for ID in IDS:
        for M in MODOS_MASCARA:
            f = ruta_arq(ID, W, M, e)
            if not os.path.exists(f):
                continue
            t = pd.read_csv(f, parse_dates=["timestamp"]).set_index("timestamp")[["Realidad", COL_PRED]].dropna()
            t = t[t["Realidad"] > TRIGGER]
            idx = t.index if idx is None else idx.intersection(t.index)
    fb, fc = os.path.join(DIR_BASE, f"test_results_{e}_baselines.csv"), ruta_chronos(W, e)
    if idx is None or not os.path.exists(fb) or not os.path.exists(fc):
        _CACHE[(W, e)] = None
        return None
    b = pd.read_csv(fb, parse_dates=["timestamp"]).set_index("timestamp")[COL_ARIMA].dropna()
    c = leer_pred(fc, "Chronos")["Chronos"].dropna()
    _CACHE[(W, e)] = idx.intersection(b.index).intersection(c.index)
    return _CACHE[(W, e)]


def datos_enlace(W, e, modelos_arq):
    """DataFrame con Realidad, ARIMA, Chronos y las columnas de modelos_arq {nombre: (ID, MODO)}."""
    idx = instantes(W, e)
    if idx is None or len(idx) == 0:
        return None
    b = pd.read_csv(os.path.join(DIR_BASE, f"test_results_{e}_baselines.csv"),
                    parse_dates=["timestamp"]).set_index("timestamp")
    j = b[["Realidad", COL_ARIMA]].rename(columns={COL_ARIMA: "ARIMA"})
    j = j.join(leer_pred(ruta_chronos(W, e), "Chronos"), how="inner")
    for nom, (ID, M) in modelos_arq.items():
        j = j.join(leer_pred(ruta_arq(ID, W, M, e), nom), how="inner")
    j = j.reindex(idx)
    if j.isna().any().any():
        print(f"⚠️ {e} W{W}: NaN dentro de los instantes comunes"); j = j.dropna()
    return j if not j.empty else None


def caracterizar(e):
    v = pd.to_numeric(pd.read_csv(os.path.join(DIR_BASE, f"test_results_{e}_baselines.csv"))["Realidad"],
                      errors="coerce").dropna()
    v = v[v > TRIGGER]
    if v.empty or v.mean() <= 0:
        return np.nan, np.nan, np.nan
    return v.std() / v.mean() * 100, v.max() / v.mean(), np.percentile(v, 99)


def evaluar(W, modelos_arq):
    """Métricas por enlace y globales para ARIMA, Chronos y modelos_arq."""
    modelos = ["ARIMA", "Chronos"] + list(modelos_arq)
    comunes = set.intersection(*[enlaces_de(ID, W, M) for ID, M in modelos_arq.values()])
    if QUITAR_INTRA:
        comunes = {e for e in comunes if len(set(e.split("_"))) > 1}       # fuera X_X
    filas, se, sy = [], {m: 0.0 for m in modelos}, 0.0
    for e in sorted(comunes):
        j = datos_enlace(W, e, modelos_arq)
        if j is None:
            continue
        y = j["Realidad"].values
        cv, pr, p99 = caracterizar(e)
        fila = {"Link": e, "CV%": cv, "PeakRatio": pr, "P99": p99}
        sy += y.sum()
        for m in modelos:
            err = y - j[m].values
            fila[("MAE", m)] = np.abs(err).mean()
            fila[("RMSE", m)] = np.sqrt((err ** 2).mean())
            fila[("MAPE", m)] = np.mean(np.abs(err) / y) * 100
            fila[("WAPE", m)] = np.abs(err).sum() / y.sum() * 100
            se[m] += np.abs(err).sum()
        filas.append(fila)
    if not filas:
        return None, None, None
    df = pd.DataFrame(filas)
    glob = {met: {m: df[(met, m)].mean() for m in modelos} for met in ["MAE", "RMSE", "MAPE"]}
    glob["WAPE"] = {m: 100 * se[m] / sy for m in modelos}
    return df, modelos, glob


def neg(t, b):
    return f"<b>{t}</b>" if b else t


def tabla_notebook(df, modelos, glob, met):
    filas = []
    for _, r in df.iterrows():
        v = {m: r[(met, m)] for m in modelos}
        mejor = min(v, key=v.get)
        filas.append({"Link": r["Link"], "CV%": f"{r['CV%']:.1f}", "PeakRatio": f"{r['PeakRatio']:.1f}",
                      "P99": sci(r["P99"]), **{m: neg(val(v[m]), m == mejor) for m in modelos}})
    g = glob[met]
    mejor = min(g, key=g.get)
    filas.append({"Link": "GLOBAL", "CV%": "", "PeakRatio": "", "P99": "",
                  **{m: neg(val(g[m]), m == mejor) for m in modelos}})
    return pd.DataFrame(filas).set_index("Link")


def latex_enlace(df, modelos, glob, met, W, MODO):
    esc = lambda s: s.replace("_", r"\_")
    cab = ["Link", r"CV\%", "PeakRatio", "P99"] + [esc(m) for m in modelos]
    L = [r"\begin{table*}[!ht]", r"\centering", r"\tiny", r"\begin{tabular}{l" + "r" * (len(cab) - 1) + "}",
         r"\toprule", " & ".join(rf"\textbf{{{c}}}" for c in cab) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        v = {m: r[(met, m)] for m in modelos}
        mejor = min(v, key=v.get)
        L.append(" & ".join([esc(r["Link"]), f"{r['CV%']:.1f}", f"{r['PeakRatio']:.1f}", sci(r["P99"])] +
                            [rf"\textbf{{{val(v[m])}}}" if m == mejor else val(v[m]) for m in modelos]) + r" \\")
    g = glob[met]
    mejor = min(g, key=g.get)
    L += [r"\midrule", " & ".join([r"\textbf{GLOBAL}", "", "", ""] +
                                  [rf"\textbf{{{val(g[m])}}}" if m == mejor else val(g[m]) for m in modelos]) + r" \\",
          r"\botrule", r"\end{tabular}", ""]
    nota = r" (GLOBAL: $\sum|e|/\sum y$ over all links)" if met == "WAPE" else ""
    unidad = r" (\%)" if met in ("MAPE", "WAPE") else ""
    trig = f"{TRIGGER:,}".replace(",", "{,}")
    L += [rf"\caption{{{met}{unidad} per link at hourly granularity for $t+1$ h, $W={W}$, variant {MODO}, and "
          rf"ground truth $> {trig}${nota}. Chronos: Chronos-Bolt (small), zero-shot, {W}-h context. "
          rf"Best model per link in bold.}}",
          rf"\label{{tab:{met.lower()}_1h_W{W}_{MODO}}}", r"\end{table*}"]
    return "\n".join(L)


def resumen_latex(R, n_enl):
    archs = [f"Arch{ID}" for ID in IDS]
    L = [r"\begin{table}[t]", r"\centering",
         rf"\caption{{Error metrics at hourly granularity ($t+1$ h) averaged over all {n_enl} links (WAPE pooled "
         r"over all links), for each input window $W$, architecture, and target formulation. MAPE and WAPE in \%. "
         r"Variant A: logarithmic difference; Variant B: logarithmic value. Chronos: Chronos-Bolt (small), zero-shot, "
         r"with a context equal to $W$. Best value per row in bold.}",
         r"\label{tab:resumen_1h}", r"\small", r"\setlength{\tabcolsep}{4pt}",
         r"\begin{tabular}{ll" + "c" * (2 + 2 * len(archs)) + "}", r"\toprule",
         r"\textbf{W} & \textbf{Metric} & \textbf{ARIMA} & \textbf{Chronos} & " +
         " & ".join(rf"\multicolumn{{2}}{{c}}{{\textbf{{{a}}}}}" for a in archs) + r" \\",
         " ".join(rf"\cmidrule(lr){{{5 + 2 * i}-{6 + 2 * i}}}" for i in range(len(archs))),
         " &  &  &  & " + " & ".join([r"\textbf{A} & \textbf{B}"] * len(archs)) + r" \\", r"\midrule"]
    for k, W in enumerate(VENTANAS):
        glob = R[W]
        orden = ["ARIMA", "Chronos"] + [f"{a}_{m}" for a in archs for m in ("A", "B")]
        for i, met in enumerate(METRICAS):
            g = glob[met]
            mejor = min((m for m in orden if m in g), key=g.get)
            celdas = [rf"\multirow{{{len(METRICAS)}}}{{*}}{{{W}}}" if i == 0 else "", met]
            celdas += [(rf"\textbf{{{val(g[m])}}}" if m == mejor else val(g[m])) if m in g else "--" for m in orden]
            L.append(" & ".join(celdas) + r" \\")
        if k < len(VENTANAS) - 1:
            L.append(r"\midrule")
    L += [r"\botrule", r"\end{tabular}", r"\end{table}"]
    return "\n".join(L)


pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)
if GUARDAR_TEX:
    os.makedirs(DIR_SALIDA, exist_ok=True)

if HACER_POR_ENLACE:
    for W in VENTANAS:
        for MODO in MODOS:
            arqs = {f"Arch{ID}_{MODO}": (ID, MODO) for ID in IDS if enlaces_de(ID, W, MODO)}
            if not arqs:
                continue
            df, modelos, glob = evaluar(W, arqs)
            if df is None:
                continue
            for met in METRICAS:
                print(f"\n===== {met} per link — t+1 h · W={W} · variante {MODO} · real > {TRIGGER} · "
                      f"{len(df)} enlaces (negrita = mejor) =====")
                display(HTML(tabla_notebook(df, modelos, glob, met).to_html(escape=False)))
                if GUARDAR_TEX:
                    with open(os.path.join(DIR_SALIDA, f"tabla_{met}_1h_W{W}_{MODO}.tex"), "w") as f:
                        f.write(latex_enlace(df, modelos, glob, met, W, MODO) + "\n")

if HACER_RESUMEN:
    R, n_enl = {}, None
    for W in VENTANAS:
        arqs = {f"Arch{ID}_{M}": (ID, M) for ID in IDS for M in ("A", "B") if enlaces_de(ID, W, M)}
        if not arqs:
            continue
        df, modelos, glob = evaluar(W, arqs)
        if df is None:
            continue
        R[W], n_enl = glob, len(df)
        print(f"\n===== RESUMEN t+1 h · W={W} · {len(df)} enlaces{' (sin intra-nodo)' if QUITAR_INTRA else ''} "
              f"(negrita = mejor de la fila) =====")
        filas = []
        for met in METRICAS:
            g = glob[met]
            mejor = min(g, key=g.get)
            filas.append({"Métrica": met, **{m: neg(val(g[m]), m == mejor) for m in modelos}})
        display(HTML(pd.DataFrame(filas).set_index("Métrica").to_html(escape=False)))
    if GUARDAR_TEX and len(R) == len(VENTANAS):
        with open(os.path.join(DIR_SALIDA, "resumen_1h.tex"), "w") as f:
            f.write(resumen_latex(R, n_enl) + "\n")
        print(f"💾 {DIR_SALIDA}/resumen_1h.tex")


===== MAE per link — t+1 h · W=24 · variante A · real > 1000 · 81 enlaces (negrita = mejor) =====


,CV%,PeakRatio,P99,ARIMA,Chronos,Arch1_A,Arch2_A,Arch3_A,Arch4_A
Link,,,,,,,,,
ADH_ADH,330.4,34.7,1.68e7,1.1e6,8.6e5,9.1e5,8.8e5,8.6e5,8.4e5
ADH_CVK,183.8,19.0,8.59e5,5.6e4,4.6e4,4.5e4,4.4e4,4.3e4,4.3e4
ADH_HTW,234.2,20.6,1.00e7,4.4e5,3.6e5,3.6e5,3.5e5,3.4e5,3.3e5
ADH_HU,225.4,37.1,8.93e5,7.7e4,7.1e4,7.0e4,6.9e4,7.0e4,7.3e4
ADH_SPK,146.4,11.4,1.61e6,1.2e5,1.1e5,9.8e4,9.4e4,9.2e4,9.4e4
ADH_TU,173.7,16.6,9.08e5,8.1e4,6.8e4,6.5e4,6.5e4,6.3e4,5.9e4
ADH_UP,105.5,10.5,5.40e4,6411,5742,5189,5039,5054,5226
ADH_WIAS,170.2,17.5,6.02e5,4.3e4,3.9e4,3.6e4,3.5e4,3.5e4,3.5e4
ADH_ZIB,200.6,24.7,5.32e6,3.7e5,2.7e5,2.7e5,2.7e5,2.6e5,2.6e5



===== RMSE per link — t+1 h · W=24 · variante A · real > 1000 · 81 enlaces (negrita = mejor) =====


,CV%,PeakRatio,P99,ARIMA,Chronos,Arch1_A,Arch2_A,Arch3_A,Arch4_A
Link,,,,,,,,,
ADH_ADH,330.4,34.7,1.68e7,3.1e6,3.0e6,3.2e6,3.1e6,3.0e6,2.9e6
ADH_CVK,183.8,19.0,8.59e5,1.4e5,1.2e5,1.3e5,1.2e5,1.2e5,1.2e5
ADH_HTW,234.2,20.6,1.00e7,9.5e5,8.9e5,1.0e6,9.9e5,9.7e5,9.6e5
ADH_HU,225.4,37.1,8.93e5,2.0e5,2.1e5,2.6e5,2.5e5,2.4e5,2.7e5
ADH_SPK,146.4,11.4,1.61e6,2.4e5,2.1e5,2.1e5,2.1e5,2.0e5,2.0e5
ADH_TU,173.7,16.6,9.08e5,1.7e5,1.6e5,1.6e5,1.5e5,1.5e5,1.4e5
ADH_UP,105.5,10.5,5.40e4,1.1e4,9994,9651,9360,9465,9427
ADH_WIAS,170.2,17.5,6.02e5,8.7e4,7.9e4,7.4e4,7.6e4,7.8e4,7.3e4
ADH_ZIB,200.6,24.7,5.32e6,9.1e5,7.5e5,8.1e5,8.3e5,7.7e5,8.2e5



===== MAPE per link — t+1 h · W=24 · variante A · real > 1000 · 81 enlaces (negrita = mejor) =====


...


===== RESUMEN t+1 h · W=24 · 81 enlaces (negrita = mejor de la fila) =====


,ARIMA,Chronos,Arch1_A,Arch1_B,Arch2_A,Arch2_B,Arch3_A,Arch3_B,Arch4_A,Arch4_B
Métrica,,,,,,,,,,
MAE,6.1e5,5.4e5,5.6e5,5.5e5,5.5e5,5.5e5,5.4e5,5.4e5,5.3e5,5.6e5
RMSE,1.2e6,1.2e6,1.3e6,1.2e6,1.2e6,1.2e6,1.2e6,1.2e6,1.2e6,1.3e6
MAPE,355.9,132.3,82.2,79.2,86.7,79.2,84.0,79.4,83.8,79.7
WAPE,46.8,41.8,43.1,42.4,42.4,42.7,41.3,42.0,40.9,43.3



===== RESUMEN t+1 h · W=168 · 81 enlaces (negrita = mejor de la fila) =====


,ARIMA,Chronos,Arch1_A,Arch1_B,Arch2_A,Arch2_B,Arch3_A,Arch3_B,Arch4_A,Arch4_B
Métrica,,,,,,,,,,
MAE,6.1e5,5.1e5,5.5e5,5.4e5,5.5e5,5.5e5,5.4e5,5.4e5,7.2e5,6.2e5
RMSE,1.2e6,1.1e6,1.2e6,1.2e6,1.2e6,1.2e6,1.2e6,1.2e6,2.6e6,1.4e6
MAPE,355.9,114.4,84.4,79.4,84.7,77.4,83.1,77.7,106.5,81.3
WAPE,46.8,39.1,42.4,41.9,42.3,42.7,41.6,41.8,55.3,48.2


💾 Tablas_latex_h_brain/resumen_1h.tex


In [1]:
# ============================================================
# Comparar_1v1_h.py — v1.1  (NIVEL HORARIO, T+1 h)
# v1.1: mismos instantes y mismas estadísticas que Tablas_por_enlace_h v1.1 (Tabla 2) y que
#       Comparar_1v1_Chronos_Base v1.7 (Tablas 5 y 6):
#       - Métrica por enlace en los INSTANTES COMUNES de cada (W, enlace): Realidad > TRIGGER y
#         predicción de las 8 arquitecturas con esa W (MODOS_MASCARA = A y B), ARIMA_t+1 y Chronos
#         con C = W. Antes cada modelo usaba sus propios instantes.
#       - CV%, PeakRatio y P99 sobre la Realidad del test con Realidad > TRIGGER:
#         CV = std/media·100, PeakRatio = max/media, P99 = percentil 99. Antes se leían de
#         brain_FINAL_1h_stats.csv (serie completa, sin umbral y PicoRatio = P99/mediana).
#       - Se imprime el WAPE GLOBAL (Σ|e|/Σy) de cada modelo para comprobar que coincide con la Tabla 2.
# v1.0: versión horaria de Comparar_1v1_Chronos_Base.py v1.6: cada arquitectura horaria (Arq1–4,
#       A y B) contra ARIMA_t+1 y contra Chronos con contexto C = W, para W = 24 y W = 168.
#       QUITAR_INTRA = True quita los pares X_X (intra-nodo).
# ============================================================
import os
import numpy as np
import pandas as pd

# ============================================================
# CONFIGURACIÓN (igual que Tablas_por_enlace_h)
# ============================================================
ROOT_RESULTS  = "Results_horas"
DIR_BASE      = f"{ROOT_RESULTS}/results_baselines_1h"
CARPETA       = "results_Arq{ID}_1h_{MODO}_W{W}"
DIR_CHRONOS   = "results_chronos-bolt-small_C{W}_1h"      # dentro de ROOT_RESULTS (contexto = W)
COL_ARIMA     = "ARIMA_t+1"
COL_PRED      = "Prediccion"
IDS           = [1, 2, 3, 4]
MODOS         = ["A", "B"]          # modelos que salen en las tablas
MODOS_MASCARA = ["A", "B"]          # modelos que fijan los instantes comunes
VENTANAS      = [24, 168]
TRIGGER       = 1000
QUITAR_INTRA  = False               # True: fuera los pares X_X (intra-nodo)
METRICA       = "WAPE"              # WAPE | MAPE | MAE
REFERENCIAS   = ["ARIMA", "Chronos"]
# ============================================================


def metrica_enlace(y, p):
    err = np.abs(y - p)
    if METRICA == "WAPE":
        return err.sum() / y.sum() * 100
    if METRICA == "MAE":
        return err.mean()
    return np.mean(err / y) * 100                        # MAPE (y > TRIGGER, nunca 0)


def es_enlace_valido(enlace):
    partes = enlace.split("_")
    return len(partes) == 2 and all(len(p) >= 2 for p in partes)


def ruta_arq(ID, W, M, e):
    n = CARPETA.format(ID=ID, W=W, MODO=M)
    return os.path.join(ROOT_RESULTS, n, f"test_{n.removeprefix('results_')}_{e}.csv")


def enlaces_chronos(W):
    d = os.path.join(ROOT_RESULTS, DIR_CHRONOS.format(W=W))
    if not os.path.isdir(d):
        raise SystemExit(f"❌ No existe {d} (ejecuta Test_Chronos_h.py)")
    pref = "test_" + os.path.basename(d).removeprefix("results_") + "_"
    enl = {f[len(pref):-4]: os.path.join(d, f) for f in sorted(os.listdir(d))
           if f.startswith(pref) and f.endswith(".csv") and es_enlace_valido(f[len(pref):-4])}
    if QUITAR_INTRA:
        enl = {e: r for e, r in enl.items() if len(set(e.split("_"))) > 1}
    return enl


def cargar_enlace(W, e, ruta_chronos):
    """Realidad + predicción de todos los modelos, en los instantes comunes de (W, enlace)."""
    j = None
    for ID in IDS:
        for M in MODOS_MASCARA:
            f = ruta_arq(ID, W, M, e)
            if not os.path.exists(f):
                continue
            t = (pd.read_csv(f, parse_dates=["timestamp"]).set_index("timestamp")[["Realidad", COL_PRED]]
                   .rename(columns={COL_PRED: f"Arq{ID}_{M}"}))
            j = t if j is None else j.join(t.drop(columns="Realidad"), how="inner")
    fb = os.path.join(DIR_BASE, f"test_results_{e}_baselines.csv")
    if j is None or not os.path.exists(fb):
        return None
    b = (pd.read_csv(fb, parse_dates=["timestamp"]).set_index("timestamp")[[COL_ARIMA]]
           .rename(columns={COL_ARIMA: "ARIMA"}))
    c = (pd.read_csv(ruta_chronos, parse_dates=["timestamp"]).set_index("timestamp")[[COL_PRED]]
           .rename(columns={COL_PRED: "Chronos"}))
    j = j.join(b, how="inner").join(c, how="inner").dropna()
    j = j[j["Realidad"] > TRIGGER]
    return j if len(j) else None


def caracterizar(e):
    """CV%, PeakRatio y P99 del tráfico real del test (Realidad > TRIGGER), como en minuto."""
    f = os.path.join(DIR_BASE, f"test_results_{e}_baselines.csv")
    v = pd.to_numeric(pd.read_csv(f)["Realidad"], errors="coerce").dropna()
    v = v[v > TRIGGER]
    if v.empty or v.mean() <= 0:
        return {"CV%": np.nan, "PicoRatio": np.nan, "P99": np.nan}
    return {"CV%": v.std() / v.mean() * 100, "PicoRatio": v.max() / v.mean(), "P99": np.percentile(v, 99)}


def stats_de_enlaces(enlaces, stats):
    sub = pd.DataFrame([stats[e] for e in enlaces]) if enlaces else pd.DataFrame()
    res = {}
    for col in ["CV%", "PicoRatio", "P99"]:
        res[f"{col}_mean"] = round(sub[col].mean(), 2) if len(sub) else None
        res[f"{col}_med"]  = round(sub[col].median(), 2) if len(sub) else None
    return res


def fmt_stat(v):
    return "N/A" if v is None else str(v)


def comparar():
    filas = []
    for W in VENTANAS:
        chr_ = enlaces_chronos(W)
        met, se, sy, stats = {}, {}, {}, {}
        for e, ruta in chr_.items():
            j = cargar_enlace(W, e, ruta)
            if j is None:
                continue
            y = j["Realidad"].values
            for m in [c for c in j.columns if c != "Realidad"]:
                p = j[m].values
                met.setdefault(m, {})[e] = metrica_enlace(y, p)
                se[m] = se.get(m, 0.0) + np.abs(y - p).sum()
                sy[m] = sy.get(m, 0.0) + y.sum()
            stats[e] = caracterizar(e)

        print(f"\n✅ W={W}: {len(stats)} enlaces en instantes comunes" + (" (sin intra-nodo)" if QUITAR_INTRA else ""))
        print("   WAPE GLOBAL (Σ|e|/Σy) — debe coincidir con la Tabla 2:")
        print("   " + " | ".join(f"{m} {100 * se[m] / sy[m]:.1f}" for m in sorted(se)))

        modelos = [f"Arq{ID}_{M}" for ID in IDS for M in MODOS if f"Arq{ID}_{M}" in met]
        for r in REFERENCIAS:
            ref = met.get(r, {})
            print(f"\n{'#'*70}")
            print(f"  t+1 h | W={W} | criterio {METRICA} | vs {r}" + (f" (Chronos C={W})" if r == "Chronos" else "")
                  + " | instantes comunes" + (" | sin intra-nodo" if QUITAR_INTRA else ""))
            print(f"{'#'*70}")
            print(f"{'Modelo':>16}  {'Gana':>5}  {'Pierde':>6}  {'Total':>6}  {'%Win':>6}  "
                  f"{'CV%_G':>7}  {'Pico_G':>7}  {'P99_G':>12}  "
                  f"{'CV%_P':>7}  {'Pico_P':>7}  {'P99_P':>12}")
            print("-" * 110)
            for mod in modelos:
                comunes = sorted(set(met[mod]) & set(ref))
                if not comunes:
                    continue
                gana   = [e for e in comunes if met[mod][e] < ref[e]]
                pierde = [e for e in comunes if met[mod][e] > ref[e]]
                empata = [e for e in comunes if met[mod][e] == ref[e]]
                total  = len(comunes)
                pct    = round(100 * len(gana) / total, 1)
                sg, sp = stats_de_enlaces(gana, stats), stats_de_enlaces(pierde, stats)
                print(f"{mod:>16}  {len(gana):>5}  {len(pierde):>6}  {total:>6}  {pct:>5.1f}%  "
                      f"{fmt_stat(sg['CV%_mean']):>7}  {fmt_stat(sg['PicoRatio_mean']):>7}  "
                      f"{fmt_stat(sg['P99_mean']):>12}  "
                      f"{fmt_stat(sp['CV%_mean']):>7}  {fmt_stat(sp['PicoRatio_mean']):>7}  "
                      f"{fmt_stat(sp['P99_mean']):>12}")
                filas.append({
                    "W": W, "Modelo": mod, "Referencia": r,
                    "N_total": total, "N_gana": len(gana), "N_pierde": len(pierde), "N_empata": len(empata),
                    "Pct_win": pct,
                    **{f"Gana_{k}": v for k, v in sg.items()},
                    **{f"Pierde_{k}": v for k, v in sp.items()},
                    "enlaces_gana": "|".join(gana), "enlaces_pierde": "|".join(pierde),
                })

    df_out = pd.DataFrame(filas)
    if not df_out.empty:
        salida = os.path.join(ROOT_RESULTS, f"1v1_h_vs_ARIMA_Chronos_{METRICA}"
                              + ("_sin_intra" if QUITAR_INTRA else "") + ".csv")
        df_out.to_csv(salida, index=False)
        print(f"\n💾 CSV guardado: {salida}")
        print("\n" + "=" * 60 + "\n  RESUMEN: % de enlaces ganados\n" + "=" * 60)
        print(df_out.pivot_table(index="Modelo", columns=["W", "Referencia"], values="Pct_win").to_string())


if __name__ == "__main__":
    comparar()


✅ W=24: 81 enlaces en instantes comunes
   WAPE GLOBAL (Σ|e|/Σy) — debe coincidir con la Tabla 2:
   ARIMA 46.8 | Arq1_A 43.1 | Arq1_B 42.4 | Arq2_A 42.4 | Arq2_B 42.7 | Arq3_A 41.3 | Arq3_B 42.0 | Arq4_A 40.9 | Arq4_B 43.3 | Chronos 41.8

######################################################################
  t+1 h | W=24 | criterio WAPE | vs ARIMA | instantes comunes
######################################################################
          Modelo   Gana  Pierde   Total    %Win    CV%_G   Pico_G         P99_G    CV%_P   Pico_P         P99_P
--------------------------------------------------------------------------------------------------------------
          Arq1_A     74       7      81   91.4%   172.77    21.34    6989970.91   141.09     9.02   23683950.35
          Arq1_B     76       5      81   93.8%    172.5    21.06    7477519.99   132.46     8.33   22950796.12
          Arq2_A     75       6      81   92.6%   172.95    21.25    6944751.34   133.49     8.11    2703152